### AIS Maritime SQL Agent (v1.1)

This script implements a foundational, stateful AI agent designed to interact with maritime AIS telemetry data stored in a PostgreSQL/PostGIS database. 

#### Current Capabilities & Limitations
* **Stateful Interaction (Conversational Memory):** The agent now utilizes LangGraph's `MemorySaver` and a message reducer to maintain persistent conversational history. By tracking sessions via a `thread_id`, it can seamlessly handle follow-up questions and understand context (e.g., remembering which specific ships were returned in previous queries).

#### Execution Flow
0. **Context Load:** LangGraph automatically retrieves and injects past `messages` based on the active session's `thread_id`.
1. **Route:** User Question + Chat History → `agent_node` 
   * *If conversational:* Generates answer → Saves AI Response to Memory → `END`
   * *If data needed:* Generates SQL → Saves User Question to Memory → `validator_node`
2. **Validate:** `validator_node` 
   * *If safe:* → `executor_node`
   * *If invalid:* → `fixer_node`
3. **Execute:** `executor_node` 
   * *If success:* Raw Data → `synthesizer_node`
   * *If DB Error:* → `fixer_node`
4. **Heal:** `fixer_node` writes new query → `validator_node` (Max 3 retries).
5. **Respond:** `synthesizer_node` → Generates Final Answer → Saves AI Response to Memory → `END`.

In [1]:
import os
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END

from typing import TypedDict, Literal, Annotated # NEW: Added Annotated
from langgraph.checkpoint.memory import MemorySaver # NEW: The memory checkpointer
from langgraph.graph.message import add_messages # NEW: The list reducer
from langchain_core.messages import HumanMessage, AIMessage # NEW: Message formatting
from langchain_core.prompts import MessagesPlaceholder # NEW: For prompt injection

from dotenv import load_dotenv
load_dotenv()

# --- PHASE 1: DATABASE CONNECTIVITY ---

def get_database_connection():
    db_user = os.getenv("DB_USER")
    db_password = os.getenv("DB_PASSWORD")
    db_host = os.getenv("DB_HOST")
    db_port = os.getenv("DB_PORT")
    db_name = os.getenv("DB_NAME")

    db_uri = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    
    return SQLDatabase.from_uri(
        db_uri,
        include_tables=['ais_data'],
        sample_rows_in_table_info=2
    )

db = get_database_connection()

# --- PHASE 2: TOOL DEFINITION ---

@tool
def execute_sql(query: str) -> str:
    """Executes a PostgreSQL query against the ais_data table and returns the result."""
    try:
        return str(db.run(query))
    except Exception as e:
        return f"ERROR: {str(e)}"

# --- PHASE 3: STATE MANAGEMENT ---

class AgentState(TypedDict, total=False):
    messages: Annotated[list, add_messages]  # <--- NEW: Tracks the conversation history safely
    question: str                
    schema_context: str          
    sql_query: str               
    validation_status: str       
    critique: str                
    query_result: str            
    final_response: str          
    retry_count: int         

# --- PHASE 4: NODE DEVELOPMENT ---

MODEL = "openai/gpt-oss-120b"
BASE_URL = "https://api.groq.com/openai/v1"

llm = ChatOpenAI(
    model=MODEL,
    base_url=BASE_URL,
    api_key=os.environ["API_KEY"],
    temperature=0.3,
)

AIS_SCHEMA_INFO = """
Table: ais_data

# COLUMN DEFINITIONS
- mmsi (text): Unique vessel identifier
- basedatetime (timestamp): Time of position report (Format: 'YYYY-MM-DDTHH:MM:SS')
- lat (float): Latitude 
- lon (float): Longitude 
- sog (float): Speed over ground in knots 
- cog (float): Course over ground in degrees 
- heading (float): True heading in degrees (0 to 359). The value 511 means 'Not Available'. 
- vesselname (text): Name of the ship (Always uppercase)
- imo (text): IMO number (Always starts with 'IMO')
- callsign (text): Call sign 
- vesseltype (integer): Numeric ITU-R M.1371 code for vessel category. 
- status (integer): Numeric ITU-R M.1371 code for navigation status. 
- length (float): Vessel length in meters 
- width (float): Vessel width in meters 
- draft (float): Vessel draft in meters 
- cargo (integer): Numeric code for cargo type.
- transceiverclass (text): AIS class (Exactly 'A' or 'B')
- geometry (geometry): PostGIS geometry column (Point)

# STATUS CODES MAPPING (`status`)
0=Moving/Under way, 1=At anchor, 2=Not under command, 3=Restricted maneuverability, 4=Constrained by draught, 5=Moored/Docked, 6=Aground, 7=Fishing, 8=Sailing, 11=Towing astern, 12=Pushing ahead/towing alongside, 14=Search and Rescue active, 15=Undefined.

# VESSEL TYPE CODES MAPPING (`vesseltype`)
30=Fishing, 31=Towing, 32=Large Towing, 33=Dredging/Underwater ops, 34=Diving ops, 35=Military ops, 36=Sailing, 37=Pleasure Craft, 40-49=High-Speed Craft (HSC), 50=Pilot Vessel, 51=Search and Rescue, 52=Tugs, 53=Port Tenders, 54=Anti-pollution, 55=Law Enforcement, 58=Medical, 60-69=Passenger Ships, 70-79=Cargo Ships, 80-89=Tankers.

# CRITICAL TRANSLATION RULES
1. When users ask for specific types of vessels or statuses, map their natural language to the integer codes. 
   - Example: 'moving cargo ships' -> `vesseltype BETWEEN 70 AND 79 AND status = 0`.
   - Example: 'parked or docked tankers' -> `vesseltype BETWEEN 80 AND 89 AND status IN (1, 5)`.
   - Example: 'ships in distress or broken down' -> `status IN (2, 6, 14)`.
2. If calculating averages, minimums, or maximums for heading, OR if filtering for valid headings, you MUST exclude 511 (e.g., `heading != 511`).

# EXAMPLES (Few-Shot Prompting)
Q: "How many Class A tankers are currently moored?"
SQL: SELECT COUNT(*) FROM ais_data WHERE transceiverclass = 'A' AND vesseltype BETWEEN 80 AND 89 AND status = 5;

Q: "Show me the names of 5 military ships that are currently moving."
SQL: SELECT vesselname FROM ais_data WHERE vesseltype = 35 AND status = 0 LIMIT 5;

Q: "Find the speed and heading of the ship named LEICESTER."
SQL: SELECT sog, heading FROM ais_data WHERE vesselname = 'LEICESTER';

Q: "What is the average heading of moving passenger ships?"
SQL: SELECT AVG(heading) FROM ais_data WHERE vesseltype BETWEEN 60 AND 69 AND status = 0 AND heading != 511;
"""
VALIDATOR_SCHEMA_INFO = """
Table: ais_data
Columns: mmsi (text), basedatetime (timestamp), lat (float), lon (float), sog (float), cog (float), heading (float, ignore 511), vesselname (text), imo (text), callsign (text), vesseltype (int), status (int), length (float), width (float), draft (float), cargo (int), transceiverclass (text, 'A' or 'B'), geometry (PostGIS point).
Valid status codes: 0, 1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15.
Valid vesseltype codes: 30-37, 40-55, 58, 60-69 (Passenger), 70-79 (Cargo), 80-89 (Tankers).
"""

def agent_node(state: AgentState):
    """Node A: The Brain (Decides whether to chat or query)"""
    print("\n--- AGENT THINKING ---")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert maritime AI assistant and PostGIS database analyst. 
        You have access to a PostgreSQL database containing AIS ship telemetry data.
        
        Database Schema & Translation Rules:
        {schema}
        
        CORE INSTRUCTIONS:
        1. WHEN TO USE THE DATABASE (TOOL USE): 
        If the user asks for specific data, statistics, vessel details, or spatial locations, you MUST use the `execute_sql` tool to generate and run a PostgreSQL query.
        - ALWAYS use 'ais_data' as the table name.
        - PROTECT THE DATABASE: Unless the user asks for an aggregation (like COUNT or AVG), always append a `LIMIT 10` (or whatever number the user specifies) to prevent massive data pulls.
        - SPATIAL AWARENESS: Use PostGIS functions (e.g., ST_DWithin, ST_MakeEnvelope) on the `geometry` column if the user asks for distance or bounding box queries.
        - NEVER hallucinate column names. Strictly adhere to the columns and definitions in the schema provided.

        2. WHEN TO CHAT (NO TOOL USE): 
        If the user says hello, asks general maritime knowledge, or explanation of a value from the AIS database (e.g., "What does AIS stand for?" or "What does the Vessel Status mean?"), or asks an unrelated question, DO NOT use the tool. Provide a helpful, natural language response directly.
        """),
        MessagesPlaceholder(variable_name="messages"), # <--- NEW: Injects past chat history right before the new question
        ("human", "{question}")
    ])
    
    # We bind the tool but DO NOT require it. The LLM chooses automatically.
    llm_with_tools = llm.bind_tools([execute_sql])
    chain = prompt | llm_with_tools
    
    # NEW: We pass the 'messages' from the state, defaulting to an empty list on the first turn
    response = chain.invoke({
        "schema": AIS_SCHEMA_INFO, 
        "messages": state.get('messages', []),
        "question": state['question']
    })
    
    # Did the LLM decide to use the database tool?
    if response.tool_calls:
        tool_call = response.tool_calls[0]
        clean_sql = tool_call['args']['query']
        print(f"Decision: Needs AIS Database query. Generated Query: {clean_sql}")
        return {
            "sql_query": clean_sql, 
            "retry_count": 0, 
            "schema_context": AIS_SCHEMA_INFO,
            "final_response": "",
            # NEW: Save the user's question to memory before moving to the database pipeline
            "messages": [HumanMessage(content=state['question'])] 
        }
    else:
        # The LLM decided to just answer conversationally
        print("Decision: Conversational reply. No database query needed.")
        return {
            "sql_query": "", 
            "retry_count": 0, 
            "schema_context": AIS_SCHEMA_INFO,
            "final_response": response.content,
            # NEW: Save both the user's question and the agent's natural answer to memory
            "messages": [
                HumanMessage(content=state['question']),
                AIMessage(content=response.content)
            ] 
        }

def validator_node(state: AgentState):
    """Node B: The Critic"""
    print("--- VALIDATING SQL ---")
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a strict PostgreSQL Security Validator.
        Review the following query for the `ais_data` table against this strict schema definition:
        
        {validation_schema}
        
        CRITICAL CHECKS:
        1. READ-ONLY: Reject if it contains DROP, DELETE, INSERT, UPDATE, ALTER.
        2. HALLUCINATIONS: Reject if it queries a column name NOT listed in the schema.
        3. DATA TYPES: Reject if `status` or `vesseltype` are queried as text strings instead of their valid integers.
        
        Output format:
        If valid: Return exactly "VALID".
        If invalid: Return a concise explanation of the exact error."""),
        ("human", "Query to check: {query}")
    ])
    
    # Pass the minified schema instead of the full generation schema
    response = (prompt | llm).invoke({
        "query": state['sql_query'],
        "validation_schema": VALIDATOR_SCHEMA_INFO 
    })
    
    status = "VALID" if "VALID" in response.content.upper() else "INVALID"
    
    if status == "INVALID":
        print(f"Validation Failed! Reason: {response.content}")
    
    return {
        "validation_status": status, 
        "critique": response.content if status == "INVALID" else ""
    }


def executor_node(state: AgentState):
    """Node C: The Executor"""
    print("--- EXECUTING SQL ---")
    
    try:
        # Attempt to run the tool
        result = execute_sql.invoke({"query": state['sql_query']})
        
        # Fixing the print statement to properly show the raw result
        print(f"Raw SQL Result: {repr(result)}")
        
        # 1. Catch expected database errors formatted by our tool
        if isinstance(result, str) and result.startswith("ERROR:"):
            print("--- DATABASE ERROR CAUGHT ---")
            return {
                "query_result": "ERROR", 
                "critique": f"Database Execution Error: {result}", 
                "validation_status": "INVALID" # Sends it to the Fixer
            }
            
        # Success path
        return {"query_result": str(result), "critique": ""}
        
    except Exception as e:
        # 2. Catch unexpected system/LangChain crashes
        print("--- CRITICAL SYSTEM ERROR CAUGHT ---")
        return {
            "query_result": "ERROR", 
            "critique": f"System Invocation Error: {str(e)}", 
            "validation_status": "INVALID" # Sends it to the Fixer
        }


def fixer_node(state: AgentState):
    """Node D: The Repairman"""
    print("--- FIXING SQL ---")
    current_retries = state.get('retry_count', 0)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an Expert PostgreSQL and PostGIS Debugger. 
        Your job is to fix a SQL query that failed during execution or validation.
        
        Original User Question: {question}
        Broken Query: {query}
        Error Message / Critique: {critique}
        
        Database Schema & Translation Rules:
        {schema}
        
        DEBUGGING CHECKLIST & RULES:
        1. DIAGNOSE: Read the error message carefully. Identify exactly why the query failed (e.g., missing column, syntax error, wrong data type).
        2. NO HALLUCINATIONS: If the error says a column does not exist, look at the schema and use the EXACT column name provided. Never invent columns.
        3. TYPE CHECK: Ensure you are using the correct data types. Remember that `vesseltype` and `status` must be queried as integers, NOT text strings.
        4. INHERIT CONSTRAINTS: The fixed query must still follow all original rules. It MUST be a read-only SELECT statement, and it MUST include a LIMIT if it is not an aggregation.
        
        Task: Rewrite the query to permanently fix the error. You MUST use the `execute_sql` tool to submit the fixed query."""),
    ])
    
    # For fixing, we DO force the tool, because we know we need a query here.
    llm_with_tools = llm.bind_tools([execute_sql], tool_choice="required")
    response = (prompt | llm_with_tools).invoke({
        "question": state['question'],
        "query": state['sql_query'],
        "critique": state['critique'],
        "schema": state['schema_context'] # Passing the rich schema here so it knows the rules
    })
    
    tool_call = response.tool_calls[0]
    fixed_sql = tool_call['args']['query']
    
    print(f"Proposed Fix: {fixed_sql}")
    
    return {"sql_query": fixed_sql, "retry_count": current_retries + 1}

def synthesizer_node(state: AgentState):
    """Node E: The Synthesizer"""
    print("--- SYNTHESIZING RESPONSE ---")
    
    if state['query_result'] == "ERROR":
        error_msg = "I was unable to retrieve that information due to database errors."
        return {
            "final_response": error_msg,
            "messages": [AIMessage(content=error_msg)] # <--- NEW: Save error reply to memory
        }
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a maritime data analyst.
        Translate the database result into a clear, natural language answer.
        Question: {question}
        Data Result: {result}
        If empty, state that no vessels matched."""),
    ])
    
    response = (prompt | llm).invoke({
        "question": state['question'],
        "result": state['query_result']
    })
    
    return {
        "final_response": response.content,
        "messages": [AIMessage(content=response.content)] # <--- NEW: Save successful AI reply to memory
    }

# --- PHASE 5: GRAPH CONSTRUCTION & ROUTING ---

# ==========================================
# 1. ROUTING FUNCTIONS 
# ==========================================
# These functions determine where the graph goes next. 
# We use 'Literal' type hints so Python (and LangGraph) knows exactly 
# which node names these functions are allowed to return.

def route_initial_query(state: AgentState) -> Literal["validator", END]:
    """
    Decides if we enter the SQL pipeline or end the conversation immediately.
    If the Agent node populated the 'sql_query' string, it means it wants to use the database.
    If 'sql_query' is empty, the Agent just answered conversationally, so we hit END.
    """
    if state.get('sql_query'):
        return "validator"
    return END

def should_continue_validation(state: AgentState) -> Literal["executor", "fixer"]:
    """
    Evaluates the Validator node's decision.
    If the SQL is safe and correct ("VALID"), route directly to the Executor node and execute the query.
    If the SQL is dangerous or hallucinates columns ("INVALID" output), route to the Fixer node to rewrite it.
    """
    return "executor" if state['validation_status'] == "VALID" else "fixer"

def should_continue_execution(state: AgentState) -> Literal["synthesizer", "fixer"]:
    """
    Evaluates the result of the actual PostgreSQL database execution, from the the executor node.
    If the database throws a syntax or execution error, catch it and send it to the Fixer node to get it fixed.
    If the query runs successfully and returns data, move to the Synthesizer node to formulate an answer.
    """
    return "fixer" if state['query_result'] == "ERROR" else "synthesizer"

def should_continue_fixing(state: AgentState) -> Literal["validator", "synthesizer"]:
    """
    Prevents infinite error loops. 
    If the Fixer node proposes a new query, we MUST send it back to the Validator node for a security check.
    However, if we've tried and failed 3 times, we give up and route to the Synthesizer 
    (which will tell the user we couldn't get the data).
    """
    return "validator" if state['retry_count'] <= 3 else "synthesizer"


# ==========================================
# 2. GRAPH INITIALIZATION
# ==========================================
# We initialize the StateGraph using our custom AgentState dictionary.
# This state dictionary will be passed to every single node.
workflow = StateGraph(AgentState)


# ==========================================
# 3. ADDING NODES
# ==========================================
# We define the "stations" in our pipeline.
# The first string is the name of the node (used for routing).
# The second argument is the actual Python function that runs at that node.
workflow.add_node("agent", agent_node)
workflow.add_node("validator", validator_node)
workflow.add_node("executor", executor_node)
workflow.add_node("fixer", fixer_node)
workflow.add_node("synthesizer", synthesizer_node)


# ==========================================
# 4. DEFINING THE EDGES
# ==========================================

# START is a special LangGraph constant. This tells the graph to always begin at the 'agent' node.
workflow.add_edge(START, "agent")

# --- CONDITIONAL EDGES (Implicit Routing) ---
# We pass the starting node and the routing function. 
# LangGraph automatically reads the string returned by the function 
# and routes the state to the node with that exact name.

# After the Agent thinks, do we run SQL or just chat?
workflow.add_conditional_edges("agent", route_initial_query)

# After the Validator checks the SQL, is it safe to run or does it need fixing?
workflow.add_conditional_edges("validator", should_continue_validation)

# After the Executor runs the SQL, did it work or did the DB throw an error?
workflow.add_conditional_edges("executor", should_continue_execution)

# After the Fixer writes a new query, send it back for validation (or give up).
workflow.add_conditional_edges("fixer", should_continue_fixing)

# --- NORMAL EDGES ---
# Once the synthesizer formulates the final natural language response, the job is completely done.
# END is a special LangGraph constant that stops the execution loop.
workflow.add_edge("synthesizer", END)

# ==========================================
# 5. COMPILATION
# ==========================================
# This freezes the graph structure and turns it into a runnable LangChain application.
app = workflow.compile()


/Users/zaidur/anaconda3/envs/lang_env/lib/python3.10/site-packages/langchain_community/utilities/sql_database.py:159: SAWarning: Did not recognize type 'geometry' of column 'geometry'
  self._metadata.reflect(


In [2]:
from langgraph.checkpoint.memory import MemorySaver

# ==========================================
# 5. COMPILATION
# ==========================================
# Initialize the in-memory checkpointer
memory = MemorySaver() 

# Compile the workflow with the checkpointer attached
app = workflow.compile(checkpointer=memory)


# ==========================================
# 6. NOTEBOOK EXECUTION LOOP
# ==========================================
# We define a configuration dictionary with a unique thread_id. 
# This acts as the session identifier so the agent remembers THIS specific conversation.
config = {"configurable": {"thread_id": "notebook_testing_session_1"}}

print("AIS Maritime Agent initialized! Type 'exit' or 'quit' to stop.")
print("-" * 60)

while True:
    # Get user input right in the notebook output cell
    user_input = input("\nYou: ")
    
    # Provide a way to break the loop
    if user_input.lower() in ['exit', 'quit']:
        print("\nAgent: Exit command received, Shutting down...")
        break
        
    # Invoke the graph with the new question and the session config
    result = app.invoke({"question": user_input}, config=config)
    
    # Print the final synthesized response
    print(f"\nAgent: {result['final_response']}")

AIS Maritime Agent initialized! Type 'exit' or 'quit' to stop.
------------------------------------------------------------

--- AGENT THINKING ---
Decision: Needs AIS Database query. Generated Query: SELECT lat, lon, basedatetime FROM ais_data WHERE vesselname = 'ADDI BELLE' ORDER BY basedatetime DESC LIMIT 1;
--- VALIDATING SQL ---
--- EXECUTING SQL ---
Raw SQL Result: '[(29.46157, -94.84932, datetime.datetime(2024, 1, 1, 23, 59, 8))]'
--- SYNTHESIZING RESPONSE ---

Agent: The most recent position recorded for the vessel **ADDI BELLE** is:

- **Latitude:** 29.46157° N  
- **Longitude:** 94.84932° W (the negative sign indicates west longitude)  
- **Timestamp:** 2024‑01‑01 23:59:08 (UTC)

So, as of the end of 2024‑01‑01, the vessel was located at approximately 29.46° N, 94.85° W.

--- AGENT THINKING ---
Decision: Needs AIS Database query. Generated Query: SELECT length, width FROM ais_data WHERE vesselname = 'ADDI BELLE' ORDER BY basedatetime DESC LIMIT 1;
--- VALIDATING SQL ---
--- E

In [ ]:
exit